# Ticket #25 — Score buzz / durable v1 : vérification

Le score est calculé par `src/sillage_score.py` avec les seuils de `config/seuils_score_v1.json` (écrits avant le test).

| Critère du ticket | Où |
|---|---|
| Au moins 2 critères jeudi (durée, géographie) | `critere_duree`, `zone_en_hausse` |
| Même entrée, même score : calcul par le code | §1 ci-dessous |
| Seuils écrits dans le dépôt avant le test sur le passé | `config/seuils_score_v1.json` (version, date, règle) |
| Ordre d'apparition et décalage entre pays (proposition) | `apparition`, champ `propagation` du résultat |

**Formule v1** : score = 0,6 × durée + 0,4 × géographie (0-100). Durée = part des 12 derniers mois où l'intérêt atteint 50 % du maximum de ces 12 mois. Géographie = part des zones où l'intérêt des 3 derniers mois dépasse 1,5 fois celui d'il y a un à deux ans. Verdict : ≥ 60 suivre, ≥ 35 surveiller, sinon ignorer.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
import pandas as pd
from sillage_score import charger_google_trends, charger_wikipedia, score, SEUILS
pd.set_option("display.width", 170)
SEUILS["_version"], SEUILS["_ecrit_le"]

('score_v1', '2026-11-12')

## 1. Déterminisme : même entrée, même score

In [2]:
s = charger_google_trends("pistachio")
r = [json.dumps(score(s, "2024-02"), sort_keys=True) for _ in range(5)]
print("5 appels identiques :", len(set(r)) == 1)
print("Le nom affiché ne change pas le score :",
      score(s, "2024-02", "Tendance A")["score"] == score(s, "2024-02", "pistache")["score"])

5 appels identiques : True


Le nom affiché ne change pas le score : True


## 2. Test sur le passé, hors des 3 cas Google Trends : 20 cas Wikipedia

Les 20 cas de `cas_reference.csv` qui ont un article Wikipedia. Date figée = 3 mois avant le pic des vues (toutes langues). Ici les « zones » sont des **langues Wikipedia, pas des pays** : la géographie est donc approximative. La pistache est exclue du calcul (label `a_tester`, non présupposé).

In [3]:
ind = pd.read_csv("data/wikipedia/indicateurs.csv", encoding="utf-8-sig")
vues = pd.read_csv("data/wikipedia/echantillon_pageviews.csv", encoding="utf-8-sig")
lignes = []
for r in ind.itertuples():
    pic = pd.Period(vues[vues.terme == r.terme].groupby("mois").vues.sum().idxmax(), "M")
    sc = score(charger_wikipedia(r.terme), str(pic - 3))
    cl = {"suivre": "tendance", "ignorer": "buzz"}.get(sc["verdict_propose"], "indécis")
    lignes.append({"terme": r.terme, "label": r.label, "date_figee": str(pic - 3), "score": sc["score"],
                   "duree": sc["composantes"]["duree"], "geographie": sc["composantes"]["geographie"],
                   "verdict": sc["verdict_propose"], "classe": cl})
ev = pd.DataFrame(lignes); ev["correct"] = ev.classe == ev.label
ev

,terme,label,date_figee,score,duree,geographie,verdict,classe,correct
0,Matcha,tendance,2025-05,69.0,75.0,60.0,suivre,tendance,True
1,Yuzu,tendance,2023-12,45.0,41.7,50.0,surveiller,indécis,False
2,Ube,tendance,2026-02,64.4,100.0,11.1,suivre,tendance,True
3,Lait d'avoine,tendance,2021-02,63.3,83.3,33.3,suivre,tendance,True
4,Kombucha,tendance,2018-05,76.0,100.0,40.0,suivre,tendance,True
5,Cold brew,tendance,2016-05,45.0,58.3,25.0,surveiller,indécis,False
6,Birria,tendance,2024-02,72.1,91.7,42.9,suivre,tendance,True
7,Gochujang,tendance,2024-10,59.0,91.7,10.0,surveiller,indécis,False
8,Hot honey,tendance,2026-05,60.0,33.3,100.0,suivre,tendance,True
9,Poke bowl,tendance,2024-05,58.9,83.3,22.2,surveiller,indécis,False


In [4]:
t = ev[ev.label != "a_tester"]
print(f"Bien classés : {t.correct.sum()} / {len(t)} ({t.correct.mean():.0%})")
print(pd.crosstab(t.label, t.verdict, margins=True))
print("\nBuzz recommandés « suivre » (l'erreur la plus coûteuse) :", int(((t.label == 'buzz') & (t.verdict == 'suivre')).sum()))

Bien classés : 14 / 20 (70%)
verdict   ignorer  suivre  surveiller  All
label                                     
buzz            5       0           1    6
tendance        1       9           4   14
All             6       9           5   20

Buzz recommandés « suivre » (l'erreur la plus coûteuse) : 0


## 3. Propagation entre pays (Google Trends, toutes données jusqu'à 2026-10)
Ordre d'apparition et décalage en mois. Un décalage est un signal à investiguer, **pas une preuve de causalité**.

In [5]:
rows = []
for t in ["matcha", "pistachio", "dalgona_coffee", "dubai_chocolate"]:
    for p in score(charger_google_trends(t), "2026-10")["propagation"]:
        rows.append({"sujet": t, **p})
pd.DataFrame(rows)

,sujet,zone,mois,decalage_mois
0,matcha,AE,2019-07,0
1,matcha,KR,2025-03,68
2,matcha,US,2025-04,69
3,pistachio,AE,2019-09,0
4,pistachio,JP,2020-11,14
5,pistachio,KR,2024-06,57
6,pistachio,FR,2025-03,66
7,dalgona_coffee,JP,2020-03,0
8,dalgona_coffee,KR,2020-03,0
9,dalgona_coffee,US,2020-03,0


## Conclusion
- **Critères du ticket remplis** : 2 critères (durée, géographie), calcul déterministe et indépendant du nom affiché, seuils versionnés dans le dépôt, propagation entre pays calculée.
- **Test hors échantillon (20 cas Wikipedia)** : 14 sur 20 bien classés (70 %). **Aucun buzz n'est recommandé « suivre »** : c'est l'erreur la plus coûteuse pour un analyste, et le score v1 l'évite sur tous les cas testés.
- **Erreurs** : surtout des tendances classées « surveiller » au lieu de « suivre » (Yuzu, Cold brew, Gochujang, Poke bowl). Le score est prudent : il préfère attendre que recommander trop tôt. Le Dubai chocolate est à 0 parce que son article Wikipedia n'existait pas encore 3 mois avant le pic : c'est une limite de la source, pas du score. Pink Sauce (buzz) passe en « surveiller », mais sa page Wikipedia est douteuse (voir rapport qualité).
- **Limite** : sur Wikipedia, les zones sont des langues, pas des pays. La géographie y est donc approximative. Le vrai test cross-market se fait sur Google Trends (pays).
- **Semaine 2** : ajouter 3 critères (catégories, lancements produits, matière première). Toute modification des seuils doit créer `score_v2`, sans retoucher v1.
